In [3]:
import json
from dataclasses import dataclass, field
from typing import Callable
from typing_extensions import Protocol

from openai import OpenAI

In [4]:
@dataclass(frozen=True, slots=True)
class LlmClient(Protocol):
    client: OpenAI
    model_name: str

    def chat(
        self,
        *,
        system_message: list[dict[str, str]],
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> str: ...

In [5]:
@dataclass(frozen=True, slots=True)
class BaseLlmClient(LlmClient):

    def _build_messages(
        self,
        system_message: list[dict[str, str]],
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> list[dict[str, str]]:
        hist = [
            {"role": h["role"], "content": h["content"]} for h in history
        ] if history else []
        return system_message + hist + messages

    def chat(
        self,
        *,
        system_message: list[dict[str, str]],
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self._build_messages(system_message, messages, history),
        )
        return response.choices[0].message.content

In [6]:
@dataclass(frozen=True)
class ToolLlmClient(LlmClient):
    tools: list[dict] = field(default_factory=list)
    tool_handlers: dict[str, Callable] = field(default_factory=dict)

    def _build_messages(
        self,
        system_message: list[dict[str, str]],
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> list[dict[str, str]]:
        hist = [
            {"role": h["role"], "content": h["content"]} for h in history
        ] if history else []
        return system_message + hist + messages

    def _handle_tool_calls(self, message) -> list[dict]:
        responses = []
        for tool_call in message.tool_calls:
            handler = self.tool_handlers.get(tool_call.function.name)
            if handler:
                args = json.loads(tool_call.function.arguments)
                result = handler(**args)
                responses.append({
                    "role": "tool",
                    "content": str(result),
                    "tool_call_id": tool_call.id,
                })
        return responses

    def chat(
        self,
        *,
        system_message: list[dict[str, str]],
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> str:
        all_messages = self._build_messages(system_message, messages, history)
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=all_messages,
            tools=self.tools,
        )

        # Loop until the model stops requesting tools
        while response.choices[0].finish_reason == "tool_calls":
            assistant_msg = response.choices[0].message
            tool_responses = self._handle_tool_calls(assistant_msg)
            all_messages.append(assistant_msg)
            all_messages.extend(tool_responses)
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=all_messages,
                tools=self.tools,
            )

        return response.choices[0].message.content

In [13]:
import sqlite3


@dataclass(frozen=True, slots=True)
class FlightDB:
    db_path: str

    def init(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "CREATE TABLE IF NOT EXISTS prices "
                "(city TEXT PRIMARY KEY, price REAL)"
            )

    def seed(self, data: dict[str, float] | None = None):
        defaults = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}
        data = data or defaults
        with sqlite3.connect(self.db_path) as conn:
            for city, price in data.items():
                conn.execute(
                    "INSERT INTO prices (city, price) VALUES (?, ?) "
                    "ON CONFLICT(city) DO UPDATE SET price = ?",
                    (city.lower(), price, price),
                )

    def get_ticket_price(self, destination_city: str) -> str:
        with sqlite3.connect(self.db_path) as conn:
            row = conn.execute(
                "SELECT price FROM prices WHERE city = ?",
                (destination_city.lower(),),
            ).fetchone()
        if row:
            return f"Ticket price to {destination_city} is ${row[0]:.0f}"
        return f"No price data available for {destination_city}"

    def set_ticket_price(self, destination_city: str, price: float) -> str:
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT INTO prices (city, price) VALUES (?, ?) "
                "ON CONFLICT(city) DO UPDATE SET price = ?",
                (destination_city.lower(), price, price),
            )
        return f"Price for {destination_city} set to ${price:.0f}"

In [ ]:
@dataclass(frozen=True, slots=True)
class FlightAI:
    db: FlightDB
    llm: ToolLlmClient
    system_message: list[dict[str, str]]

    def chat(self, message: str, history: list[dict[str, str]] | None = None) -> str:
        return self.llm.chat(
            system_message=self.system_message,
            messages=[{"role": "user", "content": message}],
            history=history,
        )

In [11]:
FLIGHT_SYSTEM_MESSAGE = [{"role": "system", "content": (
    "You are a helpful flight ticket assistant for FlightAI. "
    "You can look up ticket prices and update them. "
    "Always be polite and concise. If you don't have data for a city, say so. "
    "When setting a price, confirm the change to the user."
)}]

FLIGHT_TOOLS = [
    {"type": "function", "function": {
        "name": "get_ticket_price",
        "description": "Get the current flight ticket price to a destination city",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The destination city, e.g. 'London'"
                }
            },
            "required": ["destination_city"],
            "additionalProperties": False
        }
    }},
    {"type": "function", "function": {
        "name": "set_ticket_price",
        "description": "Set or update the flight ticket price to a destination city",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The destination city, e.g. 'London'"
                },
                "price": {
                    "type": "number",
                    "description": "The ticket price in USD"
                }
            },
            "required": ["destination_city", "price"],
            "additionalProperties": False
        }
    }},
]

In [12]:
MODEL = "gpt-4.1-mini"

# 1. Create and initialize the DB
db = FlightDB(db_path="flights.db")
db.init()
db.seed()

# 2. Create the tool-enabled LLM client, wired to DB methods
llm = ToolLlmClient(
    client=OpenAI(),
    model_name=MODEL,
    tools=FLIGHT_TOOLS,
    tool_handlers={
        "get_ticket_price": db.get_ticket_price,
        "set_ticket_price": db.set_ticket_price,
    },
)

# 3. Assemble FlightAI
flight_ai = FlightAI(db=db, llm=llm, system_message=FLIGHT_SYSTEM_MESSAGE)

# Quick test
print(flight_ai.chat("How much is a ticket to Tokyo?"))
print(flight_ai.chat("Set the price to Berlin to $650"))
print(flight_ai.chat("How much is a ticket to Berlin now?"))

The current ticket price to Tokyo is $1420. Is there anything else you would like to know?
The price for a ticket to Berlin has been set to $650. Is there anything else you would like to do?
The current ticket price to Berlin is $650. Would you like to know the prices for any other destinations or make any changes?


In [15]:
import gradio as gr

def gradio_chat(message, history):
    return flight_ai.chat(message, history)

gr.ChatInterface(fn=gradio_chat, type="messages", title="✈️ FlightAI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
